# Respaldo: Entrenamiento (Caches) para `forecasting-licores.ipynb`

Este notebook sirve para **entrenar / recalcular** modelos y métricas pesadas, de modo que el notebook principal pueda quedarse en modo *cache-only* y llegar rápido a `## Fase 9 — Forecast Futuro`.

Se ejecuta así:
- Si existen los archivos de cache esperados, **no hace nada**.
- Si falta alguno, ejecuta **todas las celdas de código** de `forecasting-licores.ipynb` hasta antes de `## Fase 9 — Forecast Futuro`.

> Nota: esta estrategia evita duplicar masivamente celdas, y te permite tener un backup operativo ahora mismo. Cuando refactorices el notebook principal (mover entrenamiento), podemos ajustar este driver para que apunte a otro origen.

In [ ]:
import json
from pathlib import Path

MAIN_NOTEBOOK = "forecasting-licores.ipynb"
models_dir = Path("artifacts") / "modeling"
sarima_cache_dir = Path("model_cache")

# ML caches
expected_ml_files = [
    models_dir / "forecasting_xgb_global.joblib",
    models_dir / "forecasting_lgb_global.joblib",
    models_dir / "forecasting_xgb_store_global.joblib",
    models_dir / "forecasting_lgb_store_global.joblib",
    models_dir / "forecasting_ensemble.json",
]

# SARIMA caches (ver load_cache(name) en el notebook principal)
expected_sarima_files = [
    sarima_cache_dir / "results_sarima_categoria.joblib",
    sarima_cache_dir / "results_sarima_tienda.joblib",
    sarima_cache_dir / "results_sarima_log.joblib",
    sarima_cache_dir / "results_sarimax_cal.joblib",
    sarima_cache_dir / "results_sarima_window.joblib",
]

expected_files = expected_ml_files + expected_sarima_files

missing = [p for p in expected_files if not p.exists()]

print("Cachés esperados:")
print(f"- ML: {len(expected_ml_files)} archivos")
print(f"- SARIMA: {len(expected_sarima_files)} archivos")

if not missing:
    print("\n✅ Todos los caches esperados existen. No se ejecuta entrenamiento.")
else:
    print("\n⚠️ Faltan caches, se ejecutará entrenamiento (celdas previas a Fase 9).")
    print("Archivos faltantes:")
    for p in missing:
        print("-", p)
    
    main_path = Path(MAIN_NOTEBOOK)
    if not main_path.exists():
        raise FileNotFoundError(f"No encuentro el notebook principal: {main_path.resolve()}")

    nb = json.loads(main_path.read_text(encoding="utf-8"))
    
    # Buscamos la primer celda cuya descripción marca el inicio de la Fase 9.
    stop_marker = "## Fase 9 — Forecast Futuro"
    stop_idx = None
    for i, cell in enumerate(nb.get("cells", [])):
        src = "".join(cell.get("source", []))
        if cell.get("cell_type") == "markdown" and stop_marker in src:
            stop_idx = i
            break

    if stop_idx is None:
        raise RuntimeError(f"No encontré el marcador '{stop_marker}' dentro de {main_path}")

    # Ejecuta SOLO las celdas de código anteriores a Fase 9.
    # Esto incluye entrenamiento + guardado de caches.
    exec_globals = globals()
    code_cells = [c for c in nb["cells"][:stop_idx] if c.get("cell_type") == "code"]
    
    print(f"\nEjecutando {len(code_cells)} celdas de código antes de Fase 9...")
    for j, cell in enumerate(code_cells, start=1):
        source = cell.get("source", [])
        code = "".join(source)
        if not code.strip():
            continue
        try:
            exec(code, exec_globals)
        except Exception as e:
            print(f"\n❌ Falló la ejecución en la celda {j}/{len(code_cells)}. Error: {type(e).__name__}: {e}")
            raise

    print("\n✅ Entrenamiento/caching completado. Verifica que existan los archivos en `artifacts/modeling/` y `model_cache/`.")


## Qué esperar

- Si las caches ya están: saldrá un mensaje `✅` y terminará rápido.
- Si faltan: se ejecutarán las celdas de entrenamiento previas a Fase 9 del notebook principal.

Después, podés correr el notebook principal hasta `## Fase 9 — Forecast Futuro` para generar los `.parquet` del forecast.